# Accent preprocessing


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
from scipy.io import wavfile
from tqdm.auto import tqdm

ROOT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'raw-data' / 'accent').is_dir())
base_dir = ROOT_DIR / 'raw-data' / 'accent'
data_dir = base_dir / 'mp3_files'
OUTPUT_DIR = base_dir / 'full-accent-audio-processed-trimmed'

SAMPLE_RATE = 16000
MAX_DURATION = 30.0
SILENCE_THRESHOLD = 20
CLIP_THRESHOLD = 1.0

## Metadata and groups

- `gender`：`male` / `female`。
- `onset_group`：`early`（english native or onset_age < 6）、`middle`（6 <= onset_age <= 12）、`late`（onset_age > 12）。
- `language_group`：`english`、`germanic`、`italic`、`baltoslavic`、`indoiranian`、`afroasiatic`、`sinotibetan`。

In [8]:
metadata_df = pd.read_excel(base_dir / 'speaker_information.xlsx')
metadata_df['filename'] = metadata_df['speech_sample'].map(lambda name: Path(name).stem)
metadata_df['is_native_english'] = metadata_df['native_language'].str.lower().eq('english')

audio_files = {''.join(p.name.split()).lower(): str(p) for p in data_dir.iterdir()
               if p.suffix.lower() in ['.mp3', '.wav']}
name_aliases = {
    'kyrgyz1.mp3': 'kirgiz1.mp3',
    'miskito6.mp3': 'mikito6.mp3',
    'pahari1.mp3': 'poonchi1.mp3',
    'hiligaynon1.mp3': 'ilonggo1.mp3',
}
metadata_df['audio_path'] = (
    metadata_df['speech_sample'].replace(name_aliases)
    .map(lambda name: audio_files.get(''.join(name.split()).lower()))
)

language_map = pd.read_csv(
    ROOT_DIR / 'dataset-preparation/audioData/accent/accent_language_mapping.csv'
).set_index('native_language')['language_group']
languages = metadata_df['native_language'].str.strip().str.lower()
metadata_df['language_group'] = languages.map(language_map)
onset = pd.to_numeric(metadata_df['onset_age'], errors='coerce')
metadata_df['onset_group'] = np.select(
    [languages.eq('english') | onset.lt(6), onset.between(6, 12), onset.gt(12)],
    ['early', 'middle', 'late'], default=None
)
metadata_df['joint_group'] = metadata_df['gender'].str.cat(
    [metadata_df['onset_group'], metadata_df['language_group']], sep='_'
)

audio_metadata = metadata_df[
    metadata_df['audio_path'].notna() & metadata_df['joint_group'].notna()
].copy()
print(f'Metadata: {len(metadata_df)}; main cohort before audio filtering: {len(audio_metadata)}')
audio_metadata[['filename', 'gender', 'onset_group', 'language_group', 'joint_group']].head()


Metadata: 3031; main cohort before audio filtering: 2327


,filename,gender,onset_group,language_group,joint_group
0,afrikaans1,female,middle,germanic,female_middle_germanic
1,afrikaans2,male,early,germanic,male_early_germanic
5,amharic1,female,early,afroasiatic,female_early_afroasiatic
6,amharic2,male,middle,afroasiatic,male_middle_afroasiatic
7,amharic3,male,middle,afroasiatic,male_middle_afroasiatic


## Audio processing

In [9]:
def process_audio_file(row):
    y, sr = librosa.load(row['audio_path'], sr=None, mono=True)
    if len(y) / sr > MAX_DURATION or y.std(dtype=np.float64) == 0:
        return

    y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE)
    if np.any(np.abs(y) >= CLIP_THRESHOLD):
        return

    y, _ = librosa.effects.trim(y, top_db=SILENCE_THRESHOLD)
    std = y.std()
    if std == 0:
        return
    y = ((y - y.mean()) / std).astype(np.float32)

    processed_filename = row['filename'] + '.wav'
    wavfile.write(OUTPUT_DIR / processed_filename, SAMPLE_RATE, y)
    result = row.to_dict()
    result.update({'processed_filename': processed_filename,
                   'length (s)': len(y) / SAMPLE_RATE, 'rate (Hz)': SAMPLE_RATE})
    return result


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in OUTPUT_DIR.glob('*.wav'):
    path.unlink()

rows = []
for _, row in tqdm(audio_metadata.iterrows(), total=len(audio_metadata), desc='Processing audio'):
    result = process_audio_file(row)
    if result is not None:
        rows.append(result)

Processing audio:   0%|          | 0/2327 [00:00<?, ?it/s]

## Export metadata

In [10]:
accent_df = pd.DataFrame(rows)
accent_df.to_csv(base_dir / 'full-accent.csv', index=False)

data_dir = OUTPUT_DIR
file_names = accent_df['processed_filename'].tolist()
file_list = [str(data_dir / name) for name in file_names]
print(f'Processed recordings: {len(accent_df)}')
print(f'WAV directory: {OUTPUT_DIR}')
accent_df.head()

Processed recordings: 1678
WAV directory: /Users/jiaxin/Documents/Summer 2025/hierarchical-bayesian-model-validation/raw-data/accent/full-accent-audio-processed-trimmed


,speakerid,speaker,native_language,alternative_native_language,city,state_or_province,country,age,gender,onset_age,...,notes,filename,is_native_english,audio_path,language_group,onset_group,joint_group,processed_filename,length (s),rate (Hz)
0,1,1,afrikaans,NaN,virginia,NaN,south africa,27.0,female,9.0,...,NaN,afrikaans1,False,/Users/jiaxin/Documents/Summer 2025/hierarchic...,germanic,middle,female_middle_germanic,afrikaans1.wav,18.752,16000
1,2,2,afrikaans,NaN,pretoria,NaN,south africa,40.0,male,5.0,...,"30 years in s. africa, 10 years in usa",afrikaans2,False,/Users/jiaxin/Documents/Summer 2025/hierarchic...,germanic,early,male_early_germanic,afrikaans2.wav,20.960,16000
2,8,3,amharic,NaN,addis ababa,NaN,ethiopia,27.0,male,8.0,...,NaN,amharic3,False,/Users/jiaxin/Documents/Summer 2025/hierarchic...,afroasiatic,middle,male_middle_afroasiatic,amharic3.wav,23.008,16000
3,10,5,amharic,NaN,addis ababa,NaN,ethiopia,31.0,male,7.0,...,NaN,amharic5,False,/Users/jiaxin/Documents/Summer 2025/hierarchic...,afroasiatic,middle,male_middle_afroasiatic,amharic5.wav,24.896,16000
4,16,2,arabic,NaN,damascus,NaN,syria,18.0,male,2.5,...,NaN,arabic2,False,/Users/jiaxin/Documents/Summer 2025/hierarchic...,afroasiatic,early,male_early_afroasiatic,arabic2.wav,25.792,16000
